# Final Phase III — CRCS Robustness & Sensitivity Study

**COLAB MASTER v3.0**

This is the final pre-specified robustness experiment following Phase II.

### Fixed grid
- Calibration: **25%, 50%, 80%**
- Poisoning prevalence: **5%, 10%, 15%, 20%**
- CRCS removal budget: **5%, 10%, 15%, 20%, 25%**
- Primary budget: **15%**, fixed before running Phase III
- Alignment/evaluation: **strict inductive only**
- Subjects: **9**
- Seeds: **5**

### Why Phase III is necessary
Phase II showed strong mean CRCS suppression, but the original 15% removal cap was
close to the 10% poisoning level and S3/S6 retained residual vulnerability. Phase III
tests whether CRCS is genuinely contamination-sensitive rather than succeeding only
because of one favorable removal cap.

### Statistical discipline
The primary family is fixed before execution:
**12 poisoned-vs-CRCS ASR tests at the 15% budget**, Holm corrected.
The 5–25% budget grid is sensitivity analysis and will not be used to redefine the
primary hypothesis after seeing the results.


In [ ]:
# STEP 1 — INSTALL COLAB DEPENDENCIES
import sys, subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "moviepy"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

packages = [
    "jedi>=0.16",
    "moabb==1.5.0",
    "mne==1.12.1",
    "numpy>=1.26",
    "scipy>=1.13",
    "pandas>=2.1",
    "scikit-learn>=1.5",
    "matplotlib>=3.8",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade-strategy", "only-if-needed", *packages],
    check=True,
)

print("✅ Dependencies installed.")
print("If Colab asks for a restart, restart once and continue from STEP 2.")


✅ Dependencies installed.
If Colab asks for a restart, restart once and continue from STEP 2.


If Colab requests a restart after STEP 1, restart once and continue from STEP 2.


In [ ]:
# STEP 2 — VERIFY ENVIRONMENT + DATASET INTERFACE
import sys
import numpy as np
import scipy
import pandas as pd
import sklearn
import matplotlib
import mne
import moabb
from moabb.datasets import BNCI2014_001

print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("SciPy        :", scipy.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)
print("Matplotlib   :", matplotlib.__version__)
print("MNE          :", mne.__version__)
print("MOABB        :", moabb.__version__)

ds = BNCI2014_001()
assert list(ds.subject_list) == list(range(1, 10))
print("✅ BNCI2014_001 subjects:", ds.subject_list)


Python       : 3.13.15
NumPy        : 2.1.3
SciPy        : 1.16.3
Pandas       : 2.2.3
Scikit-learn : 1.6.1
Matplotlib   : 3.10.0
MNE          : 1.12.1
MOABB        : 1.5.0
✅ BNCI2014_001 subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9]


In [ ]:
# STEP 3 — GENERATE THE EXACT PHASE-III PYTHON SOURCE
# No separate .py upload is required.
from pathlib import Path
import hashlib

SCRIPT = '#!/usr/bin/env python3\n"""\nPhase III robustness and sensitivity benchmark for cross-session MI-BCI backdoor defense.\n\nPurpose\n-------\nPhase II showed that calibration-referenced class-consistency screening (CRCS) can\nstrongly reduce targeted backdoor attack success while largely preserving clean\ncross-session decoding. Phase III is a pre-specified robustness study designed to\nanswer the main reviewer-critical question: does that result persist when the\npoisoning prevalence and the maximum removal budget are varied independently?\n\nDataset\n-------\nBCI Competition IV Dataset 2a (MOABB: BNCI2014_001): 9 subjects, 2 sessions,\n22 EEG channels, 250 Hz, 4 motor-imagery classes.\n\nPrimary design\n--------------\n- Strict inductive Session-1 -> Session-2 evaluation only.\n- Target calibration: 25%, 50%, 80%.\n- Poisoning rates: 5%, 10%, 15%, 20% of source-session training trials.\n- CRCS removal-budget caps: 5%, 10%, 15%, 20%, 25%.\n- 9 subjects x 5 seeds.\n- Primary pre-specified CRCS budget: 15% (carried forward from Phase II).\n- Primary inferential family: poisoned vs CRCS ASR at the 15% budget across\n  3 calibration levels x 4 poison rates = 12 paired subject-level tests,\n  Holm-corrected within that family.\n\nScientific status\n-----------------\nCRCS remains a candidate method under validation. This script does not hard-code\nperformance values and does not assume the defense succeeds.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport warnings\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import welch\nfrom scipy.integrate import trapezoid\nfrom scipy.stats import rankdata, wilcoxon\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\nfrom sklearn.metrics import (\n    accuracy_score,\n    balanced_accuracy_score,\n    cohen_kappa_score,\n    f1_score,\n)\nfrom sklearn.preprocessing import LabelEncoder\n\ntry:\n    import mne\n    from mne.decoding import CSP\n    mne.set_log_level("WARNING")\nexcept Exception as exc:  # pragma: no cover\n    raise RuntimeError(\n        "MNE is required. Install the tested environment with:\\n"\n        "  python -m pip install -r requirements_colab.txt"\n    ) from exc\n\n\n# -----------------------------------------------------------------------------\n# Reproducibility and configuration\n# -----------------------------------------------------------------------------\n\n\ndef set_global_seed(seed: int) -> None:\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n\n\n@dataclass(frozen=True)\nclass Config:\n    subjects: Tuple[int, ...] = tuple(range(1, 10))\n    seeds: Tuple[int, ...] = (11, 22, 33, 44, 55)\n    calibration_fractions: Tuple[float, ...] = (0.25, 0.50, 0.80)\n    poison_rates: Tuple[float, ...] = (0.05, 0.10, 0.15, 0.20)\n    removal_budgets: Tuple[float, ...] = (0.05, 0.10, 0.15, 0.20, 0.25)\n    primary_removal_budget: float = 0.15\n\n    fmin: float = 8.0\n    fmax: float = 30.0\n    target_class_name: str = "left_hand"\n    trigger_frequency_hz: float = 23.0\n    trigger_amplitude_std: float = 0.55\n    trigger_channels: Tuple[str, ...] = ("C3", "Cz", "C4")\n    csp_components: int = 8\n\n    crcs_min_class_samples: int = 8\n    crcs_min_reference_per_class: int = 3\n    crcs_margin_threshold: float = 0.0\n    crcs_spectral_weight: float = 0.25\n    crcs_spectral_z_threshold: float = 2.5\n    crcs_subbands: Tuple[Tuple[float, float], ...] = (\n        (8.0, 12.0),\n        (12.0, 16.0),\n        (16.0, 20.0),\n        (20.0, 24.0),\n        (24.0, 30.0),\n    )\n    output_dir: str = "bci_security_phase3_results"\n\n\ndef validate_config(cfg: Config) -> None:\n    if not cfg.subjects:\n        raise ValueError("At least one subject is required.")\n    if not cfg.seeds:\n        raise ValueError("At least one seed is required.")\n    if not all(0.0 < f < 1.0 for f in cfg.calibration_fractions):\n        raise ValueError("Phase-III calibration fractions must satisfy 0 < f < 1.")\n    if not all(0.0 < p < 0.5 for p in cfg.poison_rates):\n        raise ValueError("Poison rates must lie in (0, 0.5).")\n    if not all(0.0 < b < 0.5 for b in cfg.removal_budgets):\n        raise ValueError("Removal budgets must lie in (0, 0.5).")\n    if cfg.primary_removal_budget not in cfg.removal_budgets:\n        raise ValueError("primary_removal_budget must be present in removal_budgets.")\n    if cfg.trigger_frequency_hz <= cfg.fmin or cfg.trigger_frequency_hz >= cfg.fmax:\n        raise ValueError("Trigger frequency must lie inside the analysis passband.")\n\n\nPACKAGE_VERSION = "PHASE3-COLAB-3.0"\n\n# -----------------------------------------------------------------------------\n# Linear algebra: Euclidean Alignment (EA)\n# -----------------------------------------------------------------------------\n\n\ndef _trial_covariance(x: np.ndarray, eps: float = 1e-10) -> np.ndarray:\n    """Trace-normalized covariance of one trial, x shape = channels x samples."""\n    if x.ndim != 2:\n        raise ValueError("A trial must have shape (channels, samples).")\n    xc = x - x.mean(axis=1, keepdims=True)\n    c = (xc @ xc.T) / max(1, xc.shape[1] - 1)\n    tr = float(np.trace(c))\n    if tr <= eps:\n        c = c + eps * np.eye(c.shape[0])\n        tr = float(np.trace(c))\n    return c / tr\n\n\ndef fit_euclidean_alignment(X: np.ndarray, eps: float = 1e-7) -> np.ndarray:\n    """Fit W = R_bar^{-1/2}; X shape = trials x channels x samples."""\n    if X.ndim != 3 or len(X) < 2:\n        raise ValueError("X must be 3-D with at least two trials.")\n    covs = np.stack([_trial_covariance(x) for x in X], axis=0)\n    r_bar = covs.mean(axis=0)\n    r_bar = 0.5 * (r_bar + r_bar.T) + eps * np.eye(r_bar.shape[0])\n    vals, vecs = np.linalg.eigh(r_bar)\n    vals = np.clip(vals, eps, None)\n    inv_sqrt = (vecs * (1.0 / np.sqrt(vals))) @ vecs.T\n    return 0.5 * (inv_sqrt + inv_sqrt.T)\n\n\ndef apply_alignment(X: np.ndarray, W: np.ndarray) -> np.ndarray:\n    if X.ndim != 3 or W.ndim != 2:\n        raise ValueError("Expected X=(trials,channels,samples), W=(channels,channels).")\n    return np.einsum("ij,njt->nit", W, X, optimize=True)\n\n\n# -----------------------------------------------------------------------------\n# Targeted spectral backdoor attack\n# -----------------------------------------------------------------------------\n\n\ndef resolve_channel_indices(ch_names: Sequence[str], preferred: Sequence[str]) -> List[int]:\n    mapping = {str(name).strip().lower(): i for i, name in enumerate(ch_names)}\n    out = [mapping[name.lower()] for name in preferred if name.lower() in mapping]\n    if out:\n        return sorted(set(out))\n    # Safe fallback: central 3 channels if names are unavailable/unexpected.\n    n = len(ch_names)\n    mid = n // 2\n    return sorted(set([max(0, mid - 1), mid, min(n - 1, mid + 1)]))\n\n\ndef add_spectral_trigger(\n    X: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    frequency_hz: float,\n    amplitude_std: float,\n) -> np.ndarray:\n    """Add a deterministic narrow-band trigger scaled to each trial/channel SD."""\n    X2 = np.array(X, dtype=np.float64, copy=True)\n    n_times = X2.shape[-1]\n    t = np.arange(n_times, dtype=np.float64) / float(sfreq)\n    wave = np.sin(2.0 * np.pi * float(frequency_hz) * t)\n\n    for idx in channel_indices:\n        if idx < 0 or idx >= X2.shape[1]:\n            raise IndexError(f"Channel index {idx} out of range for {X2.shape[1]} channels.")\n        scale = np.std(X2[:, idx, :], axis=1, ddof=1)\n        scale = np.maximum(scale, np.finfo(np.float64).eps)\n        X2[:, idx, :] += amplitude_std * scale[:, None] * wave[None, :]\n    return X2\n\n\ndef poison_training_set(\n    X: np.ndarray,\n    y: np.ndarray,\n    target_class: int,\n    poison_rate: float,\n    rng: np.random.Generator,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    frequency_hz: float,\n    amplitude_std: float,\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """Dirty-label targeted poisoning: trigger non-target trials and relabel target."""\n    eligible = np.flatnonzero(y != target_class)\n    if len(eligible) == 0:\n        raise ValueError("No non-target trials are available for poisoning.")\n\n    n_poison = max(1, int(round(poison_rate * len(y))))\n    n_poison = min(n_poison, len(eligible))\n    chosen = rng.choice(eligible, size=n_poison, replace=False)\n\n    Xp = np.array(X, copy=True)\n    yp = np.array(y, copy=True)\n    Xp[chosen] = add_spectral_trigger(\n        Xp[chosen], sfreq, channel_indices, frequency_hz, amplitude_std\n    )\n    yp[chosen] = target_class\n\n    mask = np.zeros(len(y), dtype=bool)\n    mask[chosen] = True\n    return Xp, yp, mask\n\n\n# -----------------------------------------------------------------------------\n# Generic spectral-outlier sanitization baseline\n# -----------------------------------------------------------------------------\n\n\ndef spectral_concentration_score(\n    X: np.ndarray,\n    sfreq: float,\n    fmin: float,\n    fmax: float,\n    channel_indices: Optional[Sequence[int]] = None,\n) -> np.ndarray:\n    """\n    Trigger-agnostic trial score.\n\n    It does NOT use the attack frequency. For each trial it measures the strongest\n    narrow spectral concentration anywhere inside the analysis band, then takes\n    the maximum across selected channels.\n    """\n    if X.ndim != 3:\n        raise ValueError("X must be 3-D.")\n    if channel_indices is None:\n        Xi = X\n    else:\n        Xi = X[:, list(channel_indices), :]\n\n    nperseg = min(512, Xi.shape[-1])\n    freqs, psd = welch(\n        Xi,\n        fs=float(sfreq),\n        nperseg=nperseg,\n        noverlap=nperseg // 2,\n        axis=-1,\n        detrend="constant",\n        scaling="density",\n    )\n    band = (freqs >= fmin) & (freqs <= fmax)\n    if band.sum() < 3:\n        raise ValueError("Too few Welch frequency bins in the analysis band.")\n\n    p = np.maximum(psd[..., band], np.finfo(np.float64).tiny)\n    concentration = p.max(axis=-1) / p.sum(axis=-1)\n    return concentration.max(axis=1)\n\n\ndef robust_sanitize(\n    X: np.ndarray,\n    y: np.ndarray,\n    scores: np.ndarray,\n    z_threshold: float,\n    max_remove_fraction: float,\n    min_class_samples: int,\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, float]]:\n    """Remove high-score spectral outliers using median/MAD with a removal cap."""\n    if not (len(X) == len(y) == len(scores)):\n        raise ValueError("X, y and scores must have equal length.")\n\n    median = float(np.median(scores))\n    mad = float(np.median(np.abs(scores - median)))\n    robust_sigma = 1.4826 * mad\n    if robust_sigma <= np.finfo(np.float64).eps:\n        robust_z = np.zeros_like(scores, dtype=float)\n    else:\n        robust_z = (scores - median) / robust_sigma\n\n    candidates = np.flatnonzero(robust_z > z_threshold)\n    max_remove = int(math.floor(max_remove_fraction * len(y)))\n    if max_remove > 0 and len(candidates) > max_remove:\n        order = candidates[np.argsort(scores[candidates])[::-1]]\n        candidates = order[:max_remove]\n\n    keep = np.ones(len(y), dtype=bool)\n    keep[candidates] = False\n\n    # Guarantee enough examples remain in every class.\n    for cls in np.unique(y):\n        cls_idx = np.flatnonzero(y == cls)\n        kept_count = int(np.sum(keep[cls_idx]))\n        if kept_count < min_class_samples:\n            need = min(min_class_samples - kept_count, int(np.sum(~keep[cls_idx])))\n            if need > 0:\n                removed_cls = cls_idx[~keep[cls_idx]]\n                restore = removed_cls[np.argsort(scores[removed_cls])[:need]]\n                keep[restore] = True\n\n    stats = {\n        "score_median": median,\n        "score_mad": mad,\n        "removed_count": int((~keep).sum()),\n        "removed_fraction": float((~keep).mean()),\n    }\n    return X[keep], y[keep], keep, stats\n\n\n\n# -----------------------------------------------------------------------------\n# Phase-II candidate defense:\n# Calibration-Referenced Class-Consistency Screening (CRCS)\n# -----------------------------------------------------------------------------\n\n\ndef _relative_bandpower_features(\n    X: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    subbands: Sequence[Tuple[float, float]],\n) -> np.ndarray:\n    """\n    Generic trigger-agnostic descriptor.\n\n    For each selected motor channel, compute relative Welch power in fixed\n    sub-bands spanning the full 8-30 Hz MI band. The implementation never uses\n    the attack frequency. Shape: trials x (channels * subbands).\n    """\n    if X.ndim != 3:\n        raise ValueError("X must have shape (trials, channels, samples).")\n    if not channel_indices:\n        raise ValueError("At least one channel is required.")\n    Xi = X[:, list(channel_indices), :]\n    nperseg = min(512, Xi.shape[-1])\n    freqs, psd = welch(\n        Xi,\n        fs=float(sfreq),\n        nperseg=nperseg,\n        noverlap=nperseg // 2,\n        axis=-1,\n        detrend="constant",\n        scaling="density",\n    )\n    total_band = (freqs >= min(b[0] for b in subbands)) & (\n        freqs <= max(b[1] for b in subbands)\n    )\n    total = trapezoid(psd[..., total_band], freqs[total_band], axis=-1)\n    total = np.maximum(total, np.finfo(np.float64).tiny)\n\n    feats = []\n    for lo, hi in subbands:\n        mask = (freqs >= float(lo)) & (freqs < float(hi))\n        if int(mask.sum()) < 2:\n            raise ValueError(f"Too few frequency bins for CRCS subband {(lo, hi)}.")\n        bp = trapezoid(psd[..., mask], freqs[mask], axis=-1)\n        rel = np.maximum(bp / total, np.finfo(np.float64).tiny)\n        feats.append(np.log(rel))\n    # trials x channels x bands -> trials x flat features\n    F = np.stack(feats, axis=-1).reshape(len(X), -1)\n    return np.asarray(F, dtype=np.float64)\n\n\ndef _robust_location_scale(F: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:\n    med = np.median(F, axis=0)\n    mad = np.median(np.abs(F - med), axis=0)\n    scale = 1.4826 * mad\n    # Avoid zero-MAD dimensions without introducing data-dependent explosions.\n    fallback = np.std(F, axis=0, ddof=1) if len(F) > 1 else np.ones(F.shape[1])\n    scale = np.where(scale > 1e-8, scale, np.maximum(fallback, 1e-6))\n    return med, scale\n\n\ndef _standardize_with_reference(\n    F_reference: np.ndarray, F_other: np.ndarray\n) -> Tuple[np.ndarray, np.ndarray]:\n    med, scale = _robust_location_scale(F_reference)\n    return (F_reference - med) / scale, (F_other - med) / scale\n\n\ndef _class_prototypes(F: np.ndarray, y: np.ndarray) -> Dict[int, np.ndarray]:\n    protos: Dict[int, np.ndarray] = {}\n    for cls in np.unique(y):\n        idx = np.flatnonzero(y == cls)\n        if len(idx):\n            protos[int(cls)] = np.median(F[idx], axis=0)\n    return protos\n\n\ndef _prototype_distances(\n    F: np.ndarray, prototypes: Dict[int, np.ndarray]\n) -> Dict[int, np.ndarray]:\n    out: Dict[int, np.ndarray] = {}\n    for cls, proto in prototypes.items():\n        # Mean absolute robust-standardized distance is stable in small calibration sets.\n        out[int(cls)] = np.mean(np.abs(F - proto[None, :]), axis=1)\n    return out\n\n\ndef _class_mismatch_margin(\n    F: np.ndarray, y_claimed: np.ndarray, prototypes: Dict[int, np.ndarray]\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """\n    Margin > 0 means the trial is closer to another class prototype than to its\n    claimed class. This is useful for dirty-label poisoning without knowing the\n    attack target class.\n    """\n    dists = _prototype_distances(F, prototypes)\n    classes = sorted(prototypes)\n    claimed = np.full(len(F), np.nan, dtype=float)\n    best_other = np.full(len(F), np.inf, dtype=float)\n\n    for i, cls in enumerate(y_claimed):\n        c = int(cls)\n        if c not in dists:\n            continue\n        claimed[i] = dists[c][i]\n        alt = [dists[k][i] for k in classes if k != c]\n        if alt:\n            best_other[i] = float(np.min(alt))\n    margin = claimed - best_other\n    return margin, claimed, best_other\n\n\ndef calibration_referenced_consistency_screen(\n    X_source: np.ndarray,\n    y_source_claimed: np.ndarray,\n    X_reference: np.ndarray,\n    y_reference: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    subbands: Sequence[Tuple[float, float]],\n    max_remove_fraction: float,\n    min_class_samples: int,\n    min_reference_per_class: int,\n    margin_threshold: float,\n    spectral_weight: float,\n    spectral_z_threshold: float,\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, float]]:\n    """\n    CRCS candidate defense.\n\n    Reference hierarchy:\n    1) labeled clean target-session calibration data when every class has enough\n       examples;\n    2) otherwise the source set itself, using robust medians (zero-calibration\n       fallback).\n\n    The screen is attack-frequency agnostic:\n    - class-consistency term: source trial vs. class prototypes;\n    - spectral term: generic narrow-band concentration relative to the reference\n      distribution for the claimed class.\n\n    Samples are removed only when the combined evidence is suspicious, with a\n    global removal cap and per-class minimum sample guarantee.\n    """\n    if len(X_source) != len(y_source_claimed):\n        raise ValueError("Source X/y length mismatch.")\n    if len(X_reference) != len(y_reference):\n        raise ValueError("Reference X/y length mismatch.")\n\n    classes = np.unique(y_source_claimed)\n    ref_ok = len(X_reference) > 0 and all(\n        np.sum(y_reference == cls) >= min_reference_per_class for cls in classes\n    )\n    if ref_ok:\n        Xref, yref = X_reference, y_reference\n        reference_kind = "target_calibration"\n    else:\n        Xref, yref = X_source, y_source_claimed\n        reference_kind = "source_robust_fallback"\n\n    F_ref = _relative_bandpower_features(\n        Xref, sfreq, channel_indices, subbands\n    )\n    F_src = _relative_bandpower_features(\n        X_source, sfreq, channel_indices, subbands\n    )\n    F_ref_z, F_src_z = _standardize_with_reference(F_ref, F_src)\n    protos = _class_prototypes(F_ref_z, yref)\n\n    # If a class is missing from the reference, fall back to source prototypes.\n    missing = [int(c) for c in classes if int(c) not in protos]\n    if missing:\n        F_all_ref_z, F_all_src_z = _standardize_with_reference(F_src, F_src)\n        source_protos = _class_prototypes(F_all_ref_z, y_source_claimed)\n        for c in missing:\n            protos[c] = source_protos[c]\n        # Keep original source feature coordinates; missing-class fallback is rare.\n        # The cap/min-count safeguards prevent catastrophic removal.\n\n    margin, d_claim, _ = _class_mismatch_margin(\n        F_src_z, y_source_claimed, protos\n    )\n\n    # Generic spectral concentration (does not know trigger frequency).\n    src_conc = spectral_concentration_score(\n        X_source,\n        sfreq=sfreq,\n        fmin=min(b[0] for b in subbands),\n        fmax=max(b[1] for b in subbands),\n        channel_indices=channel_indices,\n    )\n    ref_conc = spectral_concentration_score(\n        Xref,\n        sfreq=sfreq,\n        fmin=min(b[0] for b in subbands),\n        fmax=max(b[1] for b in subbands),\n        channel_indices=channel_indices,\n    )\n\n    spectral_z = np.zeros(len(X_source), dtype=float)\n    for cls in classes:\n        src_i = np.flatnonzero(y_source_claimed == cls)\n        ref_i = np.flatnonzero(yref == cls)\n        if len(ref_i) < 2:\n            ref_i = np.arange(len(yref))\n        vals = ref_conc[ref_i]\n        med = float(np.median(vals))\n        mad = float(np.median(np.abs(vals - med)))\n        sigma = max(1.4826 * mad, float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0, 1e-8)\n        spectral_z[src_i] = (src_conc[src_i] - med) / sigma\n\n    # Positive class mismatch is primary; narrow-band excess is secondary.\n    positive_margin = np.maximum(margin - float(margin_threshold), 0.0)\n    spectral_excess = np.maximum(\n        spectral_z - float(spectral_z_threshold), 0.0\n    )\n    suspicion = positive_margin + float(spectral_weight) * spectral_excess\n\n    candidates = np.flatnonzero(suspicion > 0.0)\n    max_remove = int(math.floor(max_remove_fraction * len(y_source_claimed)))\n    if max_remove > 0 and len(candidates) > max_remove:\n        order = candidates[np.argsort(suspicion[candidates])[::-1]]\n        candidates = order[:max_remove]\n\n    keep = np.ones(len(y_source_claimed), dtype=bool)\n    keep[candidates] = False\n\n    # Preserve enough examples per class.\n    for cls in classes:\n        cls_idx = np.flatnonzero(y_source_claimed == cls)\n        if int(np.sum(keep[cls_idx])) < min_class_samples:\n            need = min_class_samples - int(np.sum(keep[cls_idx]))\n            removed_cls = cls_idx[~keep[cls_idx]]\n            if len(removed_cls):\n                restore = removed_cls[np.argsort(suspicion[removed_cls])[:need]]\n                keep[restore] = True\n\n    stats = {\n        "reference_target_calibration": float(reference_kind == "target_calibration"),\n        "removed_count": int(np.sum(~keep)),\n        "removed_fraction": float(np.mean(~keep)),\n        "mean_suspicion": float(np.nanmean(suspicion)),\n        "max_suspicion": float(np.nanmax(suspicion)),\n        "mean_margin": float(np.nanmean(margin)),\n        "mean_spectral_z": float(np.nanmean(spectral_z)),\n    }\n    return X_source[keep], y_source_claimed[keep], keep, stats\n\n\ndef choose_target_alignment(\n    alignment_mode: str,\n    X_source_reference: np.ndarray,\n    X_target_all: np.ndarray,\n    X_target_calibration: np.ndarray,\n) -> np.ndarray:\n    """\n    transductive: use all unlabeled target trials (Phase-I setting).\n    inductive: use calibration trials only; at 0% calibration use source alignment.\n    """\n    if alignment_mode == "transductive":\n        return fit_euclidean_alignment(X_target_all)\n    if alignment_mode == "inductive":\n        if len(X_target_calibration) >= 2:\n            return fit_euclidean_alignment(X_target_calibration)\n        return fit_euclidean_alignment(X_source_reference)\n    raise ValueError(f"Unknown alignment_mode={alignment_mode!r}")\n\n\n\n# -----------------------------------------------------------------------------\n# Classifier and metrics\n# -----------------------------------------------------------------------------\n\n\nclass CSPLDA:\n    """Small deterministic CSP + shrinkage-LDA baseline for 4-class MI."""\n\n    def __init__(self, n_components: int = 8):\n        self.n_components = int(n_components)\n        self.csp: Optional[CSP] = None\n        self.lda: Optional[LinearDiscriminantAnalysis] = None\n\n    def fit(self, X: np.ndarray, y: np.ndarray) -> "CSPLDA":\n        n_components = max(2, min(self.n_components, X.shape[1]))\n        self.csp = CSP(\n            n_components=n_components,\n            reg="ledoit_wolf",\n            log=True,\n            norm_trace=False,\n            transform_into="average_power",\n            rank=None,\n        )\n        self.lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore", category=RuntimeWarning)\n            F = self.csp.fit_transform(X, y)\n        self.lda.fit(F, y)\n        return self\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        if self.csp is None or self.lda is None:\n            raise RuntimeError("Model has not been fitted.")\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore", category=RuntimeWarning)\n            F = self.csp.transform(X)\n        return self.lda.predict(F)\n\n\ndef classification_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:\n    return {\n        "accuracy": float(accuracy_score(y_true, y_pred)),\n        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),\n        "kappa": float(cohen_kappa_score(y_true, y_pred)),\n    }\n\n\ndef attack_success_rate(\n    model: CSPLDA,\n    X_test_raw: np.ndarray,\n    y_test: np.ndarray,\n    target_class: int,\n    target_alignment: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    frequency_hz: float,\n    amplitude_std: float,\n) -> float:\n    non_target = y_test != target_class\n    if non_target.sum() == 0:\n        return float("nan")\n    X_attack = add_spectral_trigger(\n        X_test_raw[non_target],\n        sfreq,\n        channel_indices,\n        frequency_hz,\n        amplitude_std,\n    )\n    X_attack_aligned = apply_alignment(X_attack, target_alignment)\n    pred = model.predict(X_attack_aligned)\n    return float(np.mean(pred == target_class))\n\n\n# -----------------------------------------------------------------------------\n# Calibration split\n# -----------------------------------------------------------------------------\n\n\ndef stratified_calibration_split(\n    y: np.ndarray, fraction: float, rng: np.random.Generator\n) -> Tuple[np.ndarray, np.ndarray]:\n    """Return calibration/test indices; leaves at least one test trial per class."""\n    if not 0.0 <= fraction < 1.0:\n        raise ValueError("fraction must satisfy 0 <= fraction < 1.")\n\n    calib: List[int] = []\n    test: List[int] = []\n    for cls in np.unique(y):\n        idx = np.flatnonzero(y == cls)\n        idx = rng.permutation(idx)\n        if fraction == 0.0:\n            n_cal = 0\n        else:\n            n_cal = int(round(fraction * len(idx)))\n            n_cal = max(1, n_cal)\n            n_cal = min(n_cal, len(idx) - 1)\n        calib.extend(idx[:n_cal].tolist())\n        test.extend(idx[n_cal:].tolist())\n\n    calib_arr = np.asarray(sorted(calib), dtype=int)\n    test_arr = np.asarray(sorted(test), dtype=int)\n    if len(test_arr) == 0:\n        raise RuntimeError("Calibration split left no held-out target data.")\n    return calib_arr, test_arr\n\n\n# -----------------------------------------------------------------------------\n# Dataset loading\n# -----------------------------------------------------------------------------\n\n\ndef load_bnci_subject(\n    subject: int, fmin: float, fmax: float\n) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame, float, List[str]]:\n    """Load one subject through MOABB. Import is lazy for clearer install errors."""\n    try:\n        from moabb.datasets import BNCI2014_001\n        from moabb.paradigms import MotorImagery\n    except Exception as exc:  # pragma: no cover\n        raise RuntimeError(\n            "MOABB is required for dataset download/loading. Recommended:\\n"\n            "  python -m pip install -r requirements_colab.txt\\n"\n            "Use Python 3.11 or newer."\n        ) from exc\n\n    dataset = BNCI2014_001()\n    paradigm = MotorImagery(n_classes=4, fmin=fmin, fmax=fmax)\n    epochs, labels, meta = paradigm.get_data(\n        dataset=dataset,\n        subjects=[int(subject)],\n        return_epochs=True,\n    )\n\n    # MOABB/MNE returns EEG in volts. Do not rescale: CSP is scale-invariant enough\n    # for this benchmark, while trigger amplitude is defined relative to trial SD.\n    try:\n        X = epochs.get_data(copy=True)\n    except TypeError:  # compatibility with older MNE\n        X = epochs.get_data().copy()\n    y_text = np.asarray(labels).astype(str)\n    meta = pd.DataFrame(meta).reset_index(drop=True)\n    sfreq = float(epochs.info["sfreq"])\n    ch_names = list(epochs.ch_names)\n\n    if not (len(X) == len(y_text) == len(meta)):\n        raise RuntimeError("MOABB returned inconsistent data/label/metadata lengths.")\n    if "session" not in meta.columns:\n        raise RuntimeError("MOABB metadata does not contain a \'session\' column.")\n    return X, y_text, meta, sfreq, ch_names\n\n\n\n# -----------------------------------------------------------------------------\n# Phase III helpers: nested poisoning, CRCS ranking, budget application\n# -----------------------------------------------------------------------------\n\n\ndef make_nested_poison_order(y: np.ndarray, target_class: int, seed: int) -> np.ndarray:\n    """One deterministic non-target permutation; higher poison rates are nested."""\n    eligible = np.flatnonzero(y != target_class)\n    if len(eligible) == 0:\n        raise ValueError("No non-target source trials are available for poisoning.")\n    rng = np.random.default_rng(int(seed))\n    return rng.permutation(eligible)\n\n\ndef poison_training_set_from_order(\n    X: np.ndarray,\n    y: np.ndarray,\n    target_class: int,\n    poison_rate: float,\n    poison_order: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    frequency_hz: float,\n    amplitude_std: float,\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """Dirty-label attack using the first n entries of a fixed poison order."""\n    n_poison = max(1, int(round(float(poison_rate) * len(y))))\n    n_poison = min(n_poison, len(poison_order))\n    chosen = np.asarray(poison_order[:n_poison], dtype=int)\n\n    Xp = np.array(X, copy=True)\n    yp = np.array(y, copy=True)\n    Xp[chosen] = add_spectral_trigger(\n        Xp[chosen], sfreq, channel_indices, frequency_hz, amplitude_std\n    )\n    yp[chosen] = target_class\n    mask = np.zeros(len(y), dtype=bool)\n    mask[chosen] = True\n    return Xp, yp, mask\n\n\ndef crcs_suspicion_ranking(\n    X_source: np.ndarray,\n    y_source_claimed: np.ndarray,\n    X_reference: np.ndarray,\n    y_reference: np.ndarray,\n    sfreq: float,\n    channel_indices: Sequence[int],\n    subbands: Sequence[Tuple[float, float]],\n    min_reference_per_class: int,\n    margin_threshold: float,\n    spectral_weight: float,\n    spectral_z_threshold: float,\n) -> Tuple[np.ndarray, Dict[str, float]]:\n    """\n    Compute attack-frequency-agnostic CRCS suspicion once for a poisoned source set.\n    The same ranking is then evaluated under several independent removal budgets.\n    """\n    if len(X_source) != len(y_source_claimed):\n        raise ValueError("Source X/y length mismatch.")\n    if len(X_reference) != len(y_reference):\n        raise ValueError("Reference X/y length mismatch.")\n\n    classes = np.unique(y_source_claimed)\n    ref_ok = len(X_reference) > 0 and all(\n        np.sum(y_reference == cls) >= min_reference_per_class for cls in classes\n    )\n    if ref_ok:\n        Xref, yref = X_reference, y_reference\n        reference_kind = "target_calibration"\n    else:\n        Xref, yref = X_source, y_source_claimed\n        reference_kind = "source_robust_fallback"\n\n    F_ref = _relative_bandpower_features(Xref, sfreq, channel_indices, subbands)\n    F_src = _relative_bandpower_features(X_source, sfreq, channel_indices, subbands)\n    F_ref_z, F_src_z = _standardize_with_reference(F_ref, F_src)\n    protos = _class_prototypes(F_ref_z, yref)\n\n    missing = [int(c) for c in classes if int(c) not in protos]\n    if missing:\n        F_all_ref_z, _ = _standardize_with_reference(F_src, F_src)\n        source_protos = _class_prototypes(F_all_ref_z, y_source_claimed)\n        for c in missing:\n            protos[c] = source_protos[c]\n\n    margin, _, _ = _class_mismatch_margin(F_src_z, y_source_claimed, protos)\n\n    src_conc = spectral_concentration_score(\n        X_source,\n        sfreq=sfreq,\n        fmin=min(b[0] for b in subbands),\n        fmax=max(b[1] for b in subbands),\n        channel_indices=channel_indices,\n    )\n    ref_conc = spectral_concentration_score(\n        Xref,\n        sfreq=sfreq,\n        fmin=min(b[0] for b in subbands),\n        fmax=max(b[1] for b in subbands),\n        channel_indices=channel_indices,\n    )\n\n    spectral_z = np.zeros(len(X_source), dtype=float)\n    for cls in classes:\n        src_i = np.flatnonzero(y_source_claimed == cls)\n        ref_i = np.flatnonzero(yref == cls)\n        if len(ref_i) < 2:\n            ref_i = np.arange(len(yref))\n        vals = ref_conc[ref_i]\n        med = float(np.median(vals))\n        mad = float(np.median(np.abs(vals - med)))\n        sigma = max(\n            1.4826 * mad,\n            float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,\n            1e-8,\n        )\n        spectral_z[src_i] = (src_conc[src_i] - med) / sigma\n\n    positive_margin = np.maximum(margin - float(margin_threshold), 0.0)\n    spectral_excess = np.maximum(spectral_z - float(spectral_z_threshold), 0.0)\n    suspicion = positive_margin + float(spectral_weight) * spectral_excess\n\n    stats = {\n        "reference_target_calibration": float(reference_kind == "target_calibration"),\n        "candidate_fraction": float(np.mean(suspicion > 0.0)),\n        "mean_suspicion": float(np.nanmean(suspicion)),\n        "max_suspicion": float(np.nanmax(suspicion)),\n        "mean_margin": float(np.nanmean(margin)),\n        "mean_spectral_z": float(np.nanmean(spectral_z)),\n    }\n    return suspicion, stats\n\n\ndef apply_crcs_removal_budget(\n    y_source_claimed: np.ndarray,\n    suspicion: np.ndarray,\n    max_remove_fraction: float,\n    min_class_samples: int,\n) -> Tuple[np.ndarray, Dict[str, float]]:\n    """Apply one removal cap to a fixed CRCS suspicion ranking."""\n    if len(y_source_claimed) != len(suspicion):\n        raise ValueError("y/suspicion length mismatch.")\n    candidates = np.flatnonzero(suspicion > 0.0)\n    max_remove = int(math.floor(float(max_remove_fraction) * len(y_source_claimed)))\n    if max_remove > 0 and len(candidates) > max_remove:\n        order = candidates[np.argsort(suspicion[candidates])[::-1]]\n        candidates = order[:max_remove]\n\n    keep = np.ones(len(y_source_claimed), dtype=bool)\n    keep[candidates] = False\n\n    for cls in np.unique(y_source_claimed):\n        cls_idx = np.flatnonzero(y_source_claimed == cls)\n        n_kept = int(np.sum(keep[cls_idx]))\n        if n_kept < min_class_samples:\n            need = min_class_samples - n_kept\n            removed_cls = cls_idx[~keep[cls_idx]]\n            if len(removed_cls):\n                restore = removed_cls[np.argsort(suspicion[removed_cls])[:need]]\n                keep[restore] = True\n\n    return keep, {\n        "removed_count": int(np.sum(~keep)),\n        "removed_fraction": float(np.mean(~keep)),\n    }\n\n\ndef _fit_evaluate_model(\n    X_source_raw: np.ndarray,\n    y_source: np.ndarray,\n    X_cal_aligned: np.ndarray,\n    y_cal: np.ndarray,\n    X_test_raw: np.ndarray,\n    y_test: np.ndarray,\n    target_alignment: np.ndarray,\n    cfg: Config,\n    sfreq: float,\n    motor_channels: Sequence[int],\n    target_class: int,\n) -> Tuple[Dict[str, float], float]:\n    Ws = fit_euclidean_alignment(X_source_raw)\n    Xs_a = apply_alignment(X_source_raw, Ws)\n    if len(X_cal_aligned):\n        X_train = np.concatenate([Xs_a, X_cal_aligned], axis=0)\n        y_train = np.concatenate([y_source, y_cal], axis=0)\n    else:\n        X_train, y_train = Xs_a, y_source\n\n    model = CSPLDA(cfg.csp_components).fit(X_train, y_train)\n    pred = model.predict(apply_alignment(X_test_raw, target_alignment))\n    metrics = classification_metrics(y_test, pred)\n    asr = attack_success_rate(\n        model=model,\n        X_test_raw=X_test_raw,\n        y_test=y_test,\n        target_class=target_class,\n        target_alignment=target_alignment,\n        sfreq=sfreq,\n        channel_indices=motor_channels,\n        frequency_hz=cfg.trigger_frequency_hz,\n        amplitude_std=cfg.trigger_amplitude_std,\n    )\n    return metrics, asr\n\n\n# -----------------------------------------------------------------------------\n# One pre-specified Phase-III block: subject x seed x calibration\n# -----------------------------------------------------------------------------\n\n\ndef evaluate_phase3_block(\n    X: np.ndarray,\n    y: np.ndarray,\n    meta: pd.DataFrame,\n    sfreq: float,\n    ch_names: Sequence[str],\n    subject: int,\n    seed: int,\n    calibration_fraction: float,\n    target_class: int,\n    cfg: Config,\n) -> List[Dict[str, object]]:\n    sessions = sorted(meta["session"].astype(str).unique().tolist())\n    if len(sessions) < 2:\n        raise RuntimeError(f"Subject {subject}: expected >=2 sessions, got {sessions}.")\n    source_session, target_session = sessions[0], sessions[1]\n\n    src_idx = np.flatnonzero(meta["session"].astype(str).to_numpy() == source_session)\n    tgt_idx = np.flatnonzero(meta["session"].astype(str).to_numpy() == target_session)\n    Xs_raw, ys = X[src_idx], y[src_idx]\n    Xt_raw, yt = X[tgt_idx], y[tgt_idx]\n\n    rng_split = np.random.default_rng(seed)\n    cal_idx, test_idx = stratified_calibration_split(yt, calibration_fraction, rng_split)\n    X_cal_raw, y_cal = Xt_raw[cal_idx], yt[cal_idx]\n    X_test_raw, y_test = Xt_raw[test_idx], yt[test_idx]\n\n    motor_channels = resolve_channel_indices(ch_names, cfg.trigger_channels)\n\n    # Strict inductive target alignment: only labeled calibration trials are used.\n    Wt = choose_target_alignment(\n        alignment_mode="inductive",\n        X_source_reference=Xs_raw,\n        X_target_all=Xt_raw,\n        X_target_calibration=X_cal_raw,\n    )\n    Xcal_a = apply_alignment(X_cal_raw, Wt)\n\n    common = {\n        "subject": int(subject),\n        "seed": int(seed),\n        "source_session": source_session,\n        "target_session": target_session,\n        "alignment_mode": "inductive",\n        "calibration_fraction": float(calibration_fraction),\n        "n_source_original": int(len(Xs_raw)),\n        "n_target_calibration": int(len(cal_idx)),\n        "n_target_test": int(len(test_idx)),\n        "target_class": int(target_class),\n        "target_class_name": cfg.target_class_name,\n    }\n\n    rows: List[Dict[str, object]] = []\n\n    # Clean reference is fit once per block.\n    clean_metrics, clean_asr = _fit_evaluate_model(\n        Xs_raw, ys, Xcal_a, y_cal, X_test_raw, y_test, Wt,\n        cfg, sfreq, motor_channels, target_class,\n    )\n    rows.append({\n        **common,\n        "scenario": "clean",\n        "poison_rate": 0.0,\n        "removal_budget": np.nan,\n        "n_poison": 0,\n        "actual_poison_fraction": 0.0,\n        "n_source_train": int(len(Xs_raw)),\n        "asr": clean_asr,\n        **clean_metrics,\n    })\n\n    # Fixed order makes 5/10/15/20% poison sets nested for this subject/seed.\n    poison_order = make_nested_poison_order(\n        ys, target_class, seed=seed + 100_000 + 1000 * int(subject)\n    )\n\n    for poison_rate in cfg.poison_rates:\n        Xp, yp, poison_mask = poison_training_set_from_order(\n            X=Xs_raw,\n            y=ys,\n            target_class=target_class,\n            poison_rate=poison_rate,\n            poison_order=poison_order,\n            sfreq=sfreq,\n            channel_indices=motor_channels,\n            frequency_hz=cfg.trigger_frequency_hz,\n            amplitude_std=cfg.trigger_amplitude_std,\n        )\n        n_poison = int(np.sum(poison_mask))\n        actual_poison_fraction = float(n_poison / len(ys))\n\n        poison_metrics, poison_asr = _fit_evaluate_model(\n            Xp, yp, Xcal_a, y_cal, X_test_raw, y_test, Wt,\n            cfg, sfreq, motor_channels, target_class,\n        )\n        rows.append({\n            **common,\n            "scenario": "poisoned",\n            "poison_rate": float(poison_rate),\n            "removal_budget": np.nan,\n            "n_poison": n_poison,\n            "actual_poison_fraction": actual_poison_fraction,\n            "n_source_train": int(len(Xp)),\n            "asr": poison_asr,\n            **poison_metrics,\n        })\n\n        # Compute CRCS ranking once for this poison-rate/calibration block.\n        Ws_poison = fit_euclidean_alignment(Xp)\n        Xp_desc = apply_alignment(Xp, Ws_poison)\n        Xcal_desc = apply_alignment(X_cal_raw, Wt)\n        suspicion, rank_stats = crcs_suspicion_ranking(\n            X_source=Xp_desc,\n            y_source_claimed=yp,\n            X_reference=Xcal_desc,\n            y_reference=y_cal,\n            sfreq=sfreq,\n            channel_indices=motor_channels,\n            subbands=cfg.crcs_subbands,\n            min_reference_per_class=cfg.crcs_min_reference_per_class,\n            margin_threshold=cfg.crcs_margin_threshold,\n            spectral_weight=cfg.crcs_spectral_weight,\n            spectral_z_threshold=cfg.crcs_spectral_z_threshold,\n        )\n\n        for budget in cfg.removal_budgets:\n            keep, budget_stats = apply_crcs_removal_budget(\n                y_source_claimed=yp,\n                suspicion=suspicion,\n                max_remove_fraction=budget,\n                min_class_samples=cfg.crcs_min_class_samples,\n            )\n            removed = ~keep\n            poison_recall = float(np.sum(removed & poison_mask) / max(1, n_poison))\n            removal_precision = float(\n                np.sum(removed & poison_mask) / max(1, np.sum(removed))\n            )\n            clean_false_removal_rate = float(\n                np.sum(removed & ~poison_mask) / max(1, np.sum(~poison_mask))\n            )\n\n            metrics, asr = _fit_evaluate_model(\n                Xp[keep], yp[keep], Xcal_a, y_cal, X_test_raw, y_test, Wt,\n                cfg, sfreq, motor_channels, target_class,\n            )\n            rows.append({\n                **common,\n                "scenario": "crcs",\n                "poison_rate": float(poison_rate),\n                "removal_budget": float(budget),\n                "n_poison": n_poison,\n                "actual_poison_fraction": actual_poison_fraction,\n                "n_source_train": int(np.sum(keep)),\n                "defense_removed_count": int(np.sum(removed)),\n                "defense_removed_fraction": float(np.mean(removed)),\n                "defense_poison_recall": poison_recall,\n                "defense_removal_precision": removal_precision,\n                "defense_clean_false_removal_rate": clean_false_removal_rate,\n                **{f"defense_{k}": v for k, v in rank_stats.items()},\n                **{f"defense_budget_{k}": v for k, v in budget_stats.items()},\n                "asr": asr,\n                **metrics,\n            })\n\n    return rows\n\n\n# -----------------------------------------------------------------------------\n# Statistics and publication-oriented summaries\n# -----------------------------------------------------------------------------\n\n\ndef _holm_adjust(p_values: Sequence[float]) -> List[float]:\n    p = np.asarray(p_values, dtype=float)\n    if len(p) == 0:\n        return []\n    order = np.argsort(p)\n    adjusted = np.empty(len(p), dtype=float)\n    running = 0.0\n    m = len(p)\n    for rank, idx in enumerate(order):\n        val = min(1.0, (m - rank) * p[idx])\n        running = max(running, val)\n        adjusted[idx] = running\n    return adjusted.tolist()\n\n\ndef matched_rank_biserial(a: np.ndarray, b: np.ndarray) -> float:\n    """Matched-pairs rank-biserial correlation for b-a; negative favors lower b."""\n    d = np.asarray(b, float) - np.asarray(a, float)\n    d = d[np.isfinite(d) & (d != 0)]\n    if len(d) == 0:\n        return 0.0\n    r = rankdata(np.abs(d))\n    w_pos = float(np.sum(r[d > 0]))\n    w_neg = float(np.sum(r[d < 0]))\n    denom = w_pos + w_neg\n    return float((w_pos - w_neg) / denom) if denom else 0.0\n\n\ndef _paired_subject_test(\n    subject_level: pd.DataFrame,\n    calibration_fraction: float,\n    poison_rate: float,\n    scenario_a: str,\n    scenario_b: str,\n    metric: str,\n    primary_budget: float,\n) -> Optional[Dict[str, object]]:\n    block = subject_level[\n        (subject_level["calibration_fraction"] == calibration_fraction)\n        & (subject_level["poison_rate"] == poison_rate)\n    ]\n    A = block[block["scenario"] == scenario_a][["subject", metric]].rename(columns={metric: "a"})\n    if scenario_b == "crcs":\n        Bsrc = block[\n            (block["scenario"] == "crcs")\n            & np.isclose(block["removal_budget"].astype(float), primary_budget)\n        ]\n    else:\n        Bsrc = block[block["scenario"] == scenario_b]\n    B = Bsrc[["subject", metric]].rename(columns={metric: "b"})\n    pair = A.merge(B, on="subject", how="inner").dropna()\n    if len(pair) < 5:\n        return None\n    try:\n        stat, p = wilcoxon(pair["a"], pair["b"], zero_method="wilcox", alternative="two-sided")\n        stat, p = float(stat), float(p)\n    except ValueError:\n        stat, p = float("nan"), 1.0\n    diff = pair["b"].to_numpy() - pair["a"].to_numpy()\n    return {\n        "calibration_fraction": float(calibration_fraction),\n        "poison_rate": float(poison_rate),\n        "scenario_a": scenario_a,\n        "scenario_b": scenario_b,\n        "metric": metric,\n        "removal_budget_b": float(primary_budget) if scenario_b == "crcs" else np.nan,\n        "n_subjects": int(len(pair)),\n        "mean_a": float(pair["a"].mean()),\n        "mean_b": float(pair["b"].mean()),\n        "mean_difference_b_minus_a": float(np.mean(diff)),\n        "median_difference_b_minus_a": float(np.median(diff)),\n        "subjects_improved_b": int(np.sum(diff < 0)) if metric == "asr" else int(np.sum(diff > 0)),\n        "wilcoxon_stat": stat,\n        "p_raw": p,\n        "rank_biserial_b_minus_a": matched_rank_biserial(pair["a"].to_numpy(), pair["b"].to_numpy()),\n    }\n\n\ndef make_phase3_statistics(subject_level: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    """\n    Pre-specified families only.\n\n    Primary family (12 tests): poisoned vs CRCS ASR at 15% removal budget across\n    3 calibration levels x 4 poison rates. Holm correction is applied only here.\n\n    Secondary family (12 tests): clean vs CRCS accuracy at the same 15% budget.\n    """\n    primary: List[Dict[str, object]] = []\n    secondary: List[Dict[str, object]] = []\n\n    for cal in cfg.calibration_fractions:\n        for pr in cfg.poison_rates:\n            x = _paired_subject_test(\n                subject_level, cal, pr, "poisoned", "crcs", "asr", cfg.primary_removal_budget\n            )\n            if x is not None:\n                primary.append(x)\n\n            # Clean has poison_rate=0 in storage. Build pairing manually with CRCS at pr.\n            clean = subject_level[\n                (subject_level["calibration_fraction"] == cal)\n                & (subject_level["scenario"] == "clean")\n            ][["subject", "accuracy"]].rename(columns={"accuracy": "a"})\n            crcs = subject_level[\n                (subject_level["calibration_fraction"] == cal)\n                & (subject_level["poison_rate"] == pr)\n                & (subject_level["scenario"] == "crcs")\n                & np.isclose(subject_level["removal_budget"].astype(float), cfg.primary_removal_budget)\n            ][["subject", "accuracy"]].rename(columns={"accuracy": "b"})\n            pair = clean.merge(crcs, on="subject", how="inner").dropna()\n            if len(pair) >= 5:\n                try:\n                    stat, p = wilcoxon(pair["a"], pair["b"], zero_method="wilcox", alternative="two-sided")\n                    stat, p = float(stat), float(p)\n                except ValueError:\n                    stat, p = float("nan"), 1.0\n                diff = pair["b"].to_numpy() - pair["a"].to_numpy()\n                secondary.append({\n                    "calibration_fraction": float(cal),\n                    "poison_rate": float(pr),\n                    "scenario_a": "clean",\n                    "scenario_b": "crcs",\n                    "metric": "accuracy",\n                    "removal_budget_b": float(cfg.primary_removal_budget),\n                    "n_subjects": int(len(pair)),\n                    "mean_a": float(pair["a"].mean()),\n                    "mean_b": float(pair["b"].mean()),\n                    "mean_difference_b_minus_a": float(np.mean(diff)),\n                    "median_difference_b_minus_a": float(np.median(diff)),\n                    "subjects_improved_b": int(np.sum(diff > 0)),\n                    "wilcoxon_stat": stat,\n                    "p_raw": p,\n                    "rank_biserial_b_minus_a": matched_rank_biserial(pair["a"].to_numpy(), pair["b"].to_numpy()),\n                })\n\n    pcols = [\n        "calibration_fraction", "poison_rate", "scenario_a", "scenario_b", "metric",\n        "removal_budget_b", "n_subjects", "mean_a", "mean_b",\n        "mean_difference_b_minus_a", "median_difference_b_minus_a",\n        "subjects_improved_b", "wilcoxon_stat", "p_raw",\n        "p_holm_family", "rank_biserial_b_minus_a",\n    ]\n    P = pd.DataFrame(primary)\n    S = pd.DataFrame(secondary)\n    if len(P):\n        P["p_holm_family"] = _holm_adjust(P["p_raw"].to_numpy())\n        P = P[pcols]\n    else:\n        P = pd.DataFrame(columns=pcols)\n    if len(S):\n        S["p_holm_family"] = _holm_adjust(S["p_raw"].to_numpy())\n        S = S[pcols]\n    else:\n        S = pd.DataFrame(columns=pcols)\n    return P, S\n\n\ndef make_phase3_plots(raw: pd.DataFrame, out_dir: Path) -> None:\n    import matplotlib.pyplot as plt\n\n    crcs = raw[raw["scenario"] == "crcs"].copy()\n    clean = raw[raw["scenario"] == "clean"].copy()\n\n    for cal in sorted(crcs["calibration_fraction"].unique()):\n        g = crcs[crcs["calibration_fraction"] == cal]\n        agg = g.groupby(["poison_rate", "removal_budget"], as_index=False).agg(\n            asr=("asr", "mean"), accuracy=("accuracy", "mean")\n        )\n        clean_acc = float(clean[clean["calibration_fraction"] == cal]["accuracy"].mean())\n        agg["accuracy_drop_pp"] = 100.0 * (clean_acc - agg["accuracy"])\n\n        fig, ax = plt.subplots(figsize=(7.4, 4.8))\n        for pr, q in agg.groupby("poison_rate"):\n            q = q.sort_values("removal_budget")\n            ax.plot(q["removal_budget"] * 100, q["asr"] * 100, marker="o", label=f"Poison {int(round(pr*100))}%")\n        ax.axhline(15, linestyle="--", linewidth=1)\n        ax.set_xlabel("CRCS maximum removal budget (%)")\n        ax.set_ylabel("Attack success rate (%)")\n        ax.set_title(f"Phase III: ASR sensitivity — calibration {int(round(cal*100))}%")\n        ax.grid(True, alpha=0.25)\n        ax.legend()\n        fig.tight_layout()\n        fig.savefig(out_dir / f"phase3_asr_budget_calibration_{int(round(cal*100))}.png", dpi=220)\n        plt.close(fig)\n\n        fig, ax = plt.subplots(figsize=(7.4, 4.8))\n        for pr, q in agg.groupby("poison_rate"):\n            q = q.sort_values("removal_budget")\n            ax.plot(q["removal_budget"] * 100, q["accuracy_drop_pp"], marker="o", label=f"Poison {int(round(pr*100))}%")\n        ax.axhline(2, linestyle="--", linewidth=1)\n        ax.set_xlabel("CRCS maximum removal budget (%)")\n        ax.set_ylabel("Clean-reference accuracy cost (percentage points)")\n        ax.set_title(f"Phase III: utility cost — calibration {int(round(cal*100))}%")\n        ax.grid(True, alpha=0.25)\n        ax.legend()\n        fig.tight_layout()\n        fig.savefig(out_dir / f"phase3_accuracy_cost_calibration_{int(round(cal*100))}.png", dpi=220)\n        plt.close(fig)\n\n    # Hard-subject 80% calibration plots identified prospectively from Phase II.\n    for subject in (3, 6):\n        g = crcs[\n            (crcs["subject"] == subject)\n            & np.isclose(crcs["calibration_fraction"], 0.80)\n        ]\n        if len(g) == 0:\n            continue\n        agg = g.groupby(["poison_rate", "removal_budget"], as_index=False)["asr"].mean()\n        fig, ax = plt.subplots(figsize=(7.4, 4.8))\n        for pr, q in agg.groupby("poison_rate"):\n            q = q.sort_values("removal_budget")\n            ax.plot(q["removal_budget"] * 100, q["asr"] * 100, marker="o", label=f"Poison {int(round(pr*100))}%")\n        ax.axhline(15, linestyle="--", linewidth=1)\n        ax.set_xlabel("CRCS maximum removal budget (%)")\n        ax.set_ylabel("Attack success rate (%)")\n        ax.set_title(f"Hard-subject analysis: S{subject}, 80% calibration")\n        ax.grid(True, alpha=0.25)\n        ax.legend()\n        fig.tight_layout()\n        fig.savefig(out_dir / f"phase3_hard_subject_S{subject}_calibration_80.png", dpi=220)\n        plt.close(fig)\n\n\n# -----------------------------------------------------------------------------\n# Full benchmark with resumable checkpoints and ETA\n# -----------------------------------------------------------------------------\n\n\ndef _block_mask(df: pd.DataFrame, subject: int, seed: int, cal: float) -> pd.Series:\n    return (\n        (df["subject"] == int(subject))\n        & (df["seed"] == int(seed))\n        & np.isclose(df["calibration_fraction"].astype(float), float(cal))\n    )\n\n\ndef run_phase3_benchmark(cfg: Config, resume: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    validate_config(cfg)\n    out_dir = Path(cfg.output_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    checkpoint_path = out_dir / "phase3_checkpoint.csv"\n    checkpoint_config_path = out_dir / "phase3_checkpoint_config.json"\n\n    # Protect resumable runs from silent configuration mismatch.\n    cfg_signature = asdict(cfg).copy()\n    cfg_signature.pop("output_dir", None)\n    cfg_signature_text = json.dumps(cfg_signature, sort_keys=True, indent=2)\n    if resume and checkpoint_path.exists() and checkpoint_config_path.exists():\n        previous = checkpoint_config_path.read_text(encoding="utf-8")\n        if previous != cfg_signature_text:\n            raise RuntimeError(\n                "Checkpoint configuration does not match the current Phase-III grid. "\n                "Use a new output directory or remove the old checkpoint explicitly."\n            )\n    checkpoint_config_path.write_text(cfg_signature_text, encoding="utf-8")\n\n    rows_per_block = 1 + len(cfg.poison_rates) * (1 + len(cfg.removal_budgets))\n    total_blocks = len(cfg.subjects) * len(cfg.seeds) * len(cfg.calibration_fractions)\n\n    if resume and checkpoint_path.exists() and checkpoint_path.stat().st_size > 0:\n        existing = pd.read_csv(checkpoint_path)\n        print(f"Resuming from checkpoint with {len(existing)} rows.", flush=True)\n    else:\n        existing = pd.DataFrame()\n\n    completed = set()\n    if len(existing):\n        counts = existing.groupby(["subject", "seed", "calibration_fraction"]).size()\n        for key, count in counts.items():\n            if int(count) == rows_per_block:\n                completed.add((int(key[0]), int(key[1]), round(float(key[2]), 6)))\n\n    dataset_manifest: List[Dict[str, object]] = []\n    start_time = time.time()\n    processed_this_run = 0\n    total_done_initial = len(completed)\n\n    for subject in cfg.subjects:\n        print(f"\\n=== Loading subject {subject} ===", flush=True)\n        X, labels_text, meta, sfreq, ch_names = load_bnci_subject(subject, cfg.fmin, cfg.fmax)\n        le = LabelEncoder()\n        y = le.fit_transform(labels_text)\n        if cfg.target_class_name not in le.classes_:\n            raise RuntimeError(f"Target class {cfg.target_class_name!r} not found; classes={list(le.classes_)}")\n        target_class = int(le.transform([cfg.target_class_name])[0])\n        sessions = sorted(meta["session"].astype(str).unique().tolist())\n        dataset_manifest.append({\n            "subject": int(subject), "n_trials": int(len(X)), "n_channels": int(X.shape[1]),\n            "n_times": int(X.shape[2]), "sfreq": float(sfreq),\n            "sessions": "|".join(sessions), "classes": "|".join(le.classes_.tolist()),\n            "channel_names": "|".join(ch_names),\n        })\n\n        for cal in cfg.calibration_fractions:\n            for seed in cfg.seeds:\n                key = (int(subject), int(seed), round(float(cal), 6))\n                block_number = total_done_initial + processed_this_run + 1\n                if key in completed:\n                    print(\n                        f"[skip] Subject {subject} | calibration={cal:.2f} | seed={seed} "\n                        f"(checkpoint complete)", flush=True\n                    )\n                    continue\n\n                elapsed = time.time() - start_time\n                if processed_this_run > 0:\n                    sec_per_block = elapsed / processed_this_run\n                    remaining = total_blocks - (total_done_initial + processed_this_run)\n                    eta_min = sec_per_block * remaining / 60.0\n                    eta_txt = f" | ETA ~{eta_min:.1f} min"\n                else:\n                    eta_txt = ""\n                print(\n                    f"Block {block_number}/{total_blocks} | Subject {subject} | "\n                    f"calibration={cal:.2f} | seed={seed}{eta_txt}", flush=True\n                )\n\n                block_rows = evaluate_phase3_block(\n                    X=X, y=y, meta=meta, sfreq=sfreq, ch_names=ch_names,\n                    subject=subject, seed=seed, calibration_fraction=cal,\n                    target_class=target_class, cfg=cfg,\n                )\n                if len(block_rows) != rows_per_block:\n                    raise RuntimeError(\n                        f"Block produced {len(block_rows)} rows; expected {rows_per_block}."\n                    )\n\n                if len(existing):\n                    existing = existing.loc[~_block_mask(existing, subject, seed, cal)].copy()\n                existing = pd.concat([existing, pd.DataFrame(block_rows)], ignore_index=True)\n                existing.to_csv(checkpoint_path, index=False)\n                existing.to_csv(out_dir / "trial_level_results.csv", index=False)\n                processed_this_run += 1\n\n    raw = existing.copy()\n    expected_rows = total_blocks * rows_per_block\n    if len(raw) != expected_rows:\n        raise RuntimeError(f"Phase III incomplete: {len(raw)} rows, expected {expected_rows}.")\n\n    metric_cols = ["accuracy", "balanced_accuracy", "macro_f1", "kappa", "asr"]\n    summary = (\n        raw.groupby(\n            ["calibration_fraction", "scenario", "poison_rate", "removal_budget"],\n            dropna=False, as_index=False,\n        )[metric_cols]\n        .agg(["mean", "std", "count"])\n    )\n    summary.columns = [\n        "_".join([str(x) for x in col if str(x) != ""]).rstrip("_")\n        for col in summary.columns.to_flat_index()\n    ]\n    summary.to_csv(out_dir / "summary_results.csv", index=False)\n    pd.DataFrame(dataset_manifest).drop_duplicates("subject").to_csv(\n        out_dir / "dataset_manifest.csv", index=False\n    )\n    (out_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")\n\n    subject_level = (\n        raw.groupby(\n            ["subject", "calibration_fraction", "scenario", "poison_rate", "removal_budget"],\n            dropna=False, as_index=False,\n        )[metric_cols]\n        .mean()\n    )\n    subject_level.to_csv(out_dir / "subject_level_results.csv", index=False)\n\n    primary_stats, secondary_stats = make_phase3_statistics(subject_level, cfg)\n    primary_stats.to_csv(out_dir / "primary_security_statistics.csv", index=False)\n    secondary_stats.to_csv(out_dir / "secondary_utility_statistics.csv", index=False)\n\n    crcs = raw[raw["scenario"] == "crcs"].copy()\n    diag_cols = [\n        "defense_removed_fraction", "defense_poison_recall",\n        "defense_removal_precision", "defense_clean_false_removal_rate",\n        "defense_candidate_fraction", "defense_reference_target_calibration",\n    ]\n    diag = (\n        crcs.groupby(["calibration_fraction", "poison_rate", "removal_budget"], as_index=False)[diag_cols]\n        .agg(["mean", "std", "count"])\n    )\n    diag.columns = [\n        "_".join([str(x) for x in col if str(x) != ""]).rstrip("_")\n        for col in diag.columns.to_flat_index()\n    ]\n    diag.to_csv(out_dir / "defense_diagnostics.csv", index=False)\n\n    # Clean-reference utility cost and ASR reduction tables.\n    clean_ref = raw[raw["scenario"] == "clean"].groupby("calibration_fraction")[["accuracy"]].mean().rename(columns={"accuracy":"clean_accuracy"})\n    poisoned_ref = raw[raw["scenario"] == "poisoned"].groupby(["calibration_fraction", "poison_rate"])[["asr", "accuracy"]].mean().rename(columns={"asr":"poisoned_asr", "accuracy":"poisoned_accuracy"})\n    crcs_mean = crcs.groupby(["calibration_fraction", "poison_rate", "removal_budget"])[["asr", "accuracy"]].mean().rename(columns={"asr":"crcs_asr", "accuracy":"crcs_accuracy"})\n    robustness = crcs_mean.join(poisoned_ref, on=["calibration_fraction", "poison_rate"]).join(clean_ref, on="calibration_fraction").reset_index()\n    robustness["asr_absolute_reduction_pp"] = 100.0 * (robustness["poisoned_asr"] - robustness["crcs_asr"])\n    robustness["asr_relative_reduction_pct"] = 100.0 * (robustness["poisoned_asr"] - robustness["crcs_asr"]) / np.maximum(robustness["poisoned_asr"], 1e-12)\n    robustness["clean_accuracy_cost_pp"] = 100.0 * (robustness["clean_accuracy"] - robustness["crcs_accuracy"])\n    robustness.to_csv(out_dir / "robustness_summary.csv", index=False)\n\n    hard = raw[\n        raw["subject"].isin([3, 6])\n        & (raw["scenario"].isin(["poisoned", "crcs"]))\n    ].groupby(\n        ["subject", "calibration_fraction", "scenario", "poison_rate", "removal_budget"],\n        dropna=False, as_index=False,\n    )[metric_cols + ["defense_poison_recall", "defense_removal_precision", "defense_clean_false_removal_rate"]].mean()\n    hard.to_csv(out_dir / "hard_subjects_S3_S6.csv", index=False)\n\n    make_phase3_plots(raw, out_dir)\n    return raw, summary\n\n\n# -----------------------------------------------------------------------------\n# Deterministic smoke test\n# -----------------------------------------------------------------------------\n\n\ndef smoke_test() -> None:\n    print("Running Phase-III deterministic synthetic smoke test...")\n    rng = np.random.default_rng(123)\n    sfreq = 250.0\n    n_ch, n_t = 6, 500\n    y = np.repeat(np.arange(4), 24)\n    X = rng.normal(size=(len(y), n_ch, n_t))\n    # Add weak class-specific rhythms to avoid degenerate CSP.\n    t = np.arange(n_t) / sfreq\n    for cls in range(4):\n        X[y == cls, cls % n_ch, :] += 0.35 * np.sin(2*np.pi*(10+3*cls)*t)\n\n    order = make_nested_poison_order(y, target_class=0, seed=777)\n    X10, y10, m10 = poison_training_set_from_order(X, y, 0, 0.10, order, sfreq, [0,1,2], 23.0, 0.55)\n    X20, y20, m20 = poison_training_set_from_order(X, y, 0, 0.20, order, sfreq, [0,1,2], 23.0, 0.55)\n    if not np.all(m10 <= m20):\n        raise AssertionError("Nested poisoning smoke test failed.")\n\n    Xref = X[:40]\n    yref = y[:40]\n    suspicion, stats = crcs_suspicion_ranking(\n        X20, y20, Xref, yref, sfreq, [0,1,2],\n        ((8,12),(12,16),(16,20),(20,24),(24,30)),\n        min_reference_per_class=2, margin_threshold=0.0,\n        spectral_weight=0.25, spectral_z_threshold=2.5,\n    )\n    k10, _ = apply_crcs_removal_budget(y20, suspicion, 0.10, 4)\n    k25, _ = apply_crcs_removal_budget(y20, suspicion, 0.25, 4)\n    if np.sum(~k25) < np.sum(~k10):\n        raise AssertionError("Removal-budget monotonicity smoke test failed.")\n\n    model = CSPLDA(n_components=4).fit(X20[k10], y20[k10])\n    pred = model.predict(X[:20])\n    if len(pred) != 20:\n        raise AssertionError("Classifier smoke test failed.")\n    print("Smoke test PASSED.")\n\n\n# -----------------------------------------------------------------------------\n# CLI\n# -----------------------------------------------------------------------------\n\n\ndef parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--smoke-test", action="store_true")\n    p.add_argument("--quick", action="store_true")\n    p.add_argument("--resume", action="store_true", help="Resume completed blocks from checkpoint.")\n    p.add_argument("--output-dir", default="bci_security_phase3_results")\n    p.add_argument("--subjects", nargs="*", type=int, default=None)\n    p.add_argument("--seeds", nargs="*", type=int, default=None)\n    p.add_argument("--calibration", nargs="*", type=float, default=None)\n    p.add_argument("--poison-rates", nargs="*", type=float, default=None)\n    p.add_argument("--removal-budgets", nargs="*", type=float, default=None)\n    return p.parse_args(argv)\n\n\ndef main(argv: Optional[Sequence[str]] = None) -> int:\n    args = parse_args(argv)\n    if args.smoke_test:\n        smoke_test()\n        return 0\n\n    if args.quick:\n        cfg = Config(\n            subjects=(1,), seeds=(42,), calibration_fractions=(0.50, 0.80),\n            poison_rates=(0.10, 0.20), removal_budgets=(0.10, 0.15, 0.25),\n            primary_removal_budget=0.15, output_dir=args.output_dir,\n        )\n    else:\n        cfg = Config(\n            subjects=tuple(args.subjects) if args.subjects else tuple(range(1,10)),\n            seeds=tuple(args.seeds) if args.seeds else (11,22,33,44,55),\n            calibration_fractions=tuple(args.calibration) if args.calibration else (0.25,0.50,0.80),\n            poison_rates=tuple(args.poison_rates) if args.poison_rates else (0.05,0.10,0.15,0.20),\n            removal_budgets=tuple(args.removal_budgets) if args.removal_budgets else (0.05,0.10,0.15,0.20,0.25),\n            primary_removal_budget=0.15,\n            output_dir=args.output_dir,\n        )\n\n    set_global_seed(cfg.seeds[0])\n    raw, summary = run_phase3_benchmark(cfg, resume=args.resume)\n    print("\\nPhase III complete.")\n    print(f"Primary rows: {len(raw)}")\n    with pd.option_context("display.max_columns", None, "display.width", 180):\n        print(summary.to_string(index=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
path = Path('/content/bci_security_phase3_CRCS_COLAB_MASTER_v3_0.py')
path.write_text(SCRIPT, encoding='utf-8')
digest = hashlib.sha256(SCRIPT.encode('utf-8')).hexdigest()
EXPECTED = 'f42260884c84f59651cb7f11e5c395652b02bd790f0fe9798a4387b3fa563df5'
assert digest == EXPECTED, (digest, EXPECTED)
print('✅ Script created:', path)
print('✅ SHA256 verified:', digest)


✅ Script created: /content/bci_security_phase3_CRCS_COLAB_MASTER_v3_0.py
✅ SHA256 verified: f42260884c84f59651cb7f11e5c395652b02bd790f0fe9798a4387b3fa563df5


In [ ]:
# STEP 4 — COMPILE + DETERMINISTIC SMOKE TEST
!PYTHONPYCACHEPREFIX=/tmp/phase3_pycache python -m py_compile \
    /content/bci_security_phase3_CRCS_COLAB_MASTER_v3_0.py

!python /content/bci_security_phase3_CRCS_COLAB_MASTER_v3_0.py --smoke-test


Running Phase-III deterministic synthetic smoke test...
Smoke test PASSED.


In [ ]:
# STEP 5 — QUICK REAL-DATA TEST
# Subject 1, seed 42; calibration 50/80%; poison 10/20%; budgets 10/15/25%.
!python /content/bci_security_phase3_CRCS_COLAB_MASTER_v3_0.py \
    --quick \
    --output-dir /content/bci_phase3_quick_v3_0



=== Loading subject 1 ===
Choosing from all possible events
/usr/local/lib/python3.13/dist-packages/moabb/datasets/download.py:97: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100%|██████████████████████████████████████| 42.8M/42.8M [00:00<00:00, 131GB/s]
SHA256 hash of downloaded file: 054f02e70cf9c4ada1517e9b9864f45407939c1062c6793516585c6f511d0325
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS reques

In [ ]:
# STEP 6 — INSPECT QUICK RESULTS
from pathlib import Path
import pandas as pd
from IPython.display import display

out = Path("/content/bci_phase3_quick_v3_0")

for name in [
    "robustness_summary.csv",
    "defense_diagnostics.csv",
    "trial_level_results.csv",
]:
    p = out / name
    print(f"\n### {name}")
    if not p.exists() or p.stat().st_size == 0:
        print("⚠️ Not available.")
        continue
    df = pd.read_csv(p)
    display(df.head(30))

print("\n✅ Quick inspection finished.")



### robustness_summary.csv


,calibration_fraction,poison_rate,removal_budget,crcs_asr,crcs_accuracy,poisoned_asr,poisoned_accuracy,clean_accuracy,asr_absolute_reduction_pp,asr_relative_reduction_pct,clean_accuracy_cost_pp
0,0.5,0.1,0.10,0.212963,0.805556,0.666667,0.729167,0.812500,45.370370,68.055556,0.694444
1,0.5,0.1,0.15,0.000000,0.812500,0.666667,0.729167,0.812500,66.666667,100.000000,0.000000
2,0.5,0.1,0.25,0.000000,0.812500,0.666667,0.729167,0.812500,66.666667,100.000000,0.000000
3,0.5,0.2,0.10,0.666667,0.694444,0.750000,0.652778,0.812500,8.333333,11.111111,11.805556
4,0.5,0.2,0.15,0.620370,0.736111,0.750000,0.652778,0.812500,12.962963,17.283951,7.638889
5,0.5,0.2,0.25,0.000000,0.805556,0.750000,0.652778,0.812500,75.000000,100.000000,0.694444
6,0.8,0.1,0.10,0.333333,0.821429,0.380952,0.767857,0.803571,4.761905,12.500000,-1.785714
7,0.8,0.1,0.15,0.000000,0.803571,0.380952,0.767857,0.803571,38.095238,100.000000,0.000000
8,0.8,0.1,0.25,0.000000,0.821429,0.380952,0.767857,0.803571,38.095238,100.000000,-1.785714
9,0.8,0.2,0.10,0.476190,0.750000,0.476190,0.696429,0.803571,0.000000,0.000000,5.357143



### defense_diagnostics.csv


,calibration_fraction,poison_rate,removal_budget,defense_removed_fraction_mean,defense_removed_fraction_std,defense_removed_fraction_count,defense_poison_recall_mean,defense_poison_recall_std,defense_poison_recall_count,defense_removal_precision_mean,...,defense_removal_precision_count,defense_clean_false_removal_rate_mean,defense_clean_false_removal_rate_std,defense_clean_false_removal_rate_count,defense_candidate_fraction_mean,defense_candidate_fraction_std,defense_candidate_fraction_count,defense_reference_target_calibration_mean,defense_reference_target_calibration_std,defense_reference_target_calibration_count
0,0.5,0.1,0.10,0.097222,NaN,1,0.931034,NaN,1,0.964286,...,1,0.003861,NaN,1,0.552083,NaN,1,1.0,NaN,1
1,0.5,0.1,0.15,0.149306,NaN,1,1.000000,NaN,1,0.674419,...,1,0.054054,NaN,1,0.552083,NaN,1,1.0,NaN,1
2,0.5,0.1,0.25,0.250000,NaN,1,1.000000,NaN,1,0.402778,...,1,0.166023,NaN,1,0.552083,NaN,1,1.0,NaN,1
3,0.5,0.2,0.10,0.097222,NaN,1,0.465517,NaN,1,0.964286,...,1,0.004348,NaN,1,0.614583,NaN,1,1.0,NaN,1
4,0.5,0.2,0.15,0.149306,NaN,1,0.706897,NaN,1,0.953488,...,1,0.008696,NaN,1,0.614583,NaN,1,1.0,NaN,1
5,0.5,0.2,0.25,0.250000,NaN,1,0.982759,NaN,1,0.791667,...,1,0.065217,NaN,1,0.614583,NaN,1,1.0,NaN,1
6,0.8,0.1,0.10,0.097222,NaN,1,0.931034,NaN,1,0.964286,...,1,0.003861,NaN,1,0.555556,NaN,1,1.0,NaN,1
7,0.8,0.1,0.15,0.149306,NaN,1,1.000000,NaN,1,0.674419,...,1,0.054054,NaN,1,0.555556,NaN,1,1.0,NaN,1
8,0.8,0.1,0.25,0.250000,NaN,1,1.000000,NaN,1,0.402778,...,1,0.166023,NaN,1,0.555556,NaN,1,1.0,NaN,1
9,0.8,0.2,0.10,0.097222,NaN,1,0.465517,NaN,1,0.964286,...,1,0.004348,NaN,1,0.618056,NaN,1,1.0,NaN,1



### trial_level_results.csv


,subject,seed,source_session,target_session,alignment_mode,calibration_fraction,n_source_original,n_target_calibration,n_target_test,target_class,...,defense_removal_precision,defense_clean_false_removal_rate,defense_reference_target_calibration,defense_candidate_fraction,defense_mean_suspicion,defense_max_suspicion,defense_mean_margin,defense_mean_spectral_z,defense_budget_removed_count,defense_budget_removed_fraction
0,1,42,0train,1test,inductive,0.5,288,144,144,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,42,0train,1test,inductive,0.5,288,144,144,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,42,0train,1test,inductive,0.5,288,144,144,1,...,0.964286,0.003861,1.0,0.552083,0.183491,1.563350,0.035023,1.050912,28.0,0.097222
3,1,42,0train,1test,inductive,0.5,288,144,144,1,...,0.674419,0.054054,1.0,0.552083,0.183491,1.563350,0.035023,1.050912,43.0,0.149306
4,1,42,0train,1test,inductive,0.5,288,144,144,1,...,0.402778,0.166023,1.0,0.552083,0.183491,1.563350,0.035023,1.050912,72.0,0.250000
5,1,42,0train,1test,inductive,0.5,288,144,144,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,42,0train,1test,inductive,0.5,288,144,144,1,...,0.964286,0.004348,1.0,0.614583,0.235421,1.589640,0.059826,1.389809,28.0,0.097222
7,1,42,0train,1test,inductive,0.5,288,144,144,1,...,0.953488,0.008696,1.0,0.614583,0.235421,1.589640,0.059826,1.389809,43.0,0.149306
8,1,42,0train,1test,inductive,0.5,288,144,144,1,...,0.791667,0.065217,1.0,0.614583,0.235421,1.589640,0.059826,1.389809,72.0,0.250000
9,1,42,0train,1test,inductive,0.8,288,232,56,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



✅ Quick inspection finished.


In [ ]:
# STEP 7 — MOUNT GOOGLE DRIVE FOR PERSISTENT CHECKPOINTING
# Strongly recommended because the full Phase-III run is longer than Phase II.
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

PHASE3_OUT = "/content/drive/MyDrive/BCI_PHASE3_CRCS_v3_0"
Path(PHASE3_OUT).mkdir(parents=True, exist_ok=True)
os.environ["PHASE3_OUT"] = PHASE3_OUT

print("✅ Persistent output directory:", PHASE3_OUT)
print("If Colab disconnects, remount Drive, rerun STEP 3 if needed,")
print("then rerun STEP 8. --resume will skip completed blocks.")


Mounted at /content/drive
✅ Persistent output directory: /content/drive/MyDrive/BCI_PHASE3_CRCS_v3_0
If Colab disconnects, remount Drive, rerun STEP 3 if needed,
then rerun STEP 8. --resume will skip completed blocks.


In [ ]:
# STEP 8 — FULL FINAL PHASE-III BENCHMARK
# 135 resumable blocks; 3375 primary rows.
# Progress output reports block number and an ETA after the first completed block.
!python /content/bci_security_phase3_CRCS_COLAB_MASTER_v3_0.py \
    --resume \
    --output-dir "$PHASE3_OUT"



=== Loading subject 1 ===
Choosing from all possible events
Block 1/135 | Subject 1 | calibration=0.25 | seed=11
Block 2/135 | Subject 1 | calibration=0.25 | seed=22 | ETA ~152.8 min
Block 3/135 | Subject 1 | calibration=0.25 | seed=33 | ETA ~139.3 min
Block 4/135 | Subject 1 | calibration=0.25 | seed=44 | ETA ~136.4 min
Block 5/135 | Subject 1 | calibration=0.25 | seed=55 | ETA ~133.8 min
Block 6/135 | Subject 1 | calibration=0.50 | seed=11 | ETA ~132.3 min
Block 7/135 | Subject 1 | calibration=0.50 | seed=22 | ETA ~132.2 min
Block 8/135 | Subject 1 | calibration=0.50 | seed=33 | ETA ~133.0 min
Block 9/135 | Subject 1 | calibration=0.50 | seed=44 | ETA ~132.4 min
Block 10/135 | Subject 1 | calibration=0.50 | seed=55 | ETA ~132.4 min
Block 11/135 | Subject 1 | calibration=0.80 | seed=11 | ETA ~131.5 min
Block 12/135 | Subject 1 | calibration=0.80 | seed=22 | ETA ~132.7 min
Block 13/135 | Subject 1 | calibration=0.80 | seed=33 | ETA ~133.5 min
Block 14/135 | Subject 1 | calibration=0.8

In [ ]:
# STEP 9 — COMPLETENESS + PRE-SPECIFIED PRIMARY RESULTS
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

out = Path(os.environ["PHASE3_OUT"])

raw = pd.read_csv(out / "trial_level_results.csv")
primary = pd.read_csv(out / "primary_security_statistics.csv")
secondary = pd.read_csv(out / "secondary_utility_statistics.csv")
robustness = pd.read_csv(out / "robustness_summary.csv")
hard = pd.read_csv(out / "hard_subjects_S3_S6.csv")

expected = 3375
print("Rows obtained :", len(raw))
print("Rows expected :", expected)
assert len(raw) == expected

assert sorted(raw["calibration_fraction"].unique().round(2).tolist()) == [0.25, 0.5, 0.8]
assert sorted(raw.loc[raw["scenario"] != "clean", "poison_rate"].unique().round(2).tolist()) == [0.05, 0.1, 0.15, 0.2]
assert sorted(raw.loc[raw["scenario"] == "crcs", "removal_budget"].unique().round(2).tolist()) == [0.05, 0.1, 0.15, 0.2, 0.25]
assert sorted(raw["subject"].unique().tolist()) == list(range(1, 10))

print("\n✅ Complete Phase-III matrix.")
print("\n### PRIMARY SECURITY FAMILY — fixed 15% CRCS budget")
display(primary)

print("\n### SECONDARY UTILITY FAMILY")
display(secondary)

print("\n### Robustness grid — selected columns")
display(
    robustness[
        [
            "calibration_fraction", "poison_rate", "removal_budget",
            "poisoned_asr", "crcs_asr", "asr_absolute_reduction_pp",
            "clean_accuracy", "crcs_accuracy", "clean_accuracy_cost_pp",
        ]
    ]
)

print("\n### Pre-identified difficult subjects S3 and S6")
display(hard.head(100))


In [ ]:
# STEP 10 — PACKAGE AND DOWNLOAD THE COMPLETE PHASE-III RESULTS
from pathlib import Path
import shutil, os
from google.colab import files

result_dir = Path(os.environ["PHASE3_OUT"])

archive = shutil.make_archive(
    "/content/BCI_SECURITY_PHASE3_CRCS_v3_0_FULL_RESULTS",
    "zip",
    root_dir=result_dir,
)

print("✅ Results archive:", archive)
files.download(archive)
